# Appendix — Data Verification Figures

Supplementary figures for the data characterisation section (sec1).
Saved to `thesis_figures/sec_appendix/`.

In [ ]:
import sys, os

os.chdir("/home/bobby/repos/latent-neural-dynamics-modeling")
sys.path.insert(0, ".")
sys.path.insert(0, "notebooks")

In [ ]:
from collections import namedtuple, defaultdict
from pathlib import Path
import numpy as np
import polars as pl
import yaml
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.lines import Line2D
from scipy.signal import welch
from scipy.stats import mannwhitneyu, gaussian_kde

from modules.style import (
    COLOR_DBS_OFF,
    COLOR_DBS_ON,
    COLOR_DPAD,
    COLOR_PSID,
    apply_thesis_style,
    hex_to_rgba,
    panel_label,
    stack_bar_label,
)
from modules.loaders import (
    discover_session_run,
    EXP_Z_AS_BEHAVIOR,
    SESSIONS,
)
from modules.lib.fig_grid_alignment import _load_trial_row, _raw_and_interp_signal

apply_thesis_style()

In [ ]:
OUT = Path("thesis_figures/sec_appendix")
OUT.mkdir(parents=True, exist_ok=True)
results_root = Path("results").resolve()
raw_data_root = Path(
    "resampled_recordings/participants_at_200Hz_scaled_1e6_narrow_band"
)
ECOG_CHANNELS_LABELS = ["ECOG_1", "ECOG_2", "ECOG_3", "ECOG_4"]
p2_root = Path("data/participants_2")

In [ ]:
# Session registry — auto-discovered via discover_session_run.
SessionSpec = namedtuple(
    "SessionSpec", ["label", "participant", "session", "psid_variant", "psid_run_ts"]
)

session_specs = []
for name in SESSIONS:
    var, ts = discover_session_run(results_root, "psid", EXP_Z_AS_BEHAVIOR, name)
    if not var:
        continue
    pid, sess = name.split("_S")
    session_specs.append(
        SessionSpec(
            label=name,
            participant=pid,
            session=int(sess),
            psid_variant=var,
            psid_run_ts=ts,
        )
    )

In [ ]:
p2_root = Path("data/participants_2")


def _p2_block_info(pid, ses):

In [ ]:
# Split manifest CSV — derived from split parquets (no configs/splits YAML needed).
splits_cfg_dir = None  # unused, kept for reference
manifest_rows = []
for s in session_specs:
    fw = s.psid_variant.split("_")[0]
    base = results_root / fw / s.psid_variant / "split"
    split_dfs = {
        sp: pl.read_parquet(base / f"{sp}.parquet", columns=["block", "stim"])
        for sp in ("train", "val", "test")
    }

    def _blocks(sp):
        return sorted(split_dfs[sp]["block"].unique().to_list())

    def _n(sp, stim):
        return split_dfs[sp].filter(pl.col("stim") == stim).height

    manifest_rows.append(
        {
            "session": s.label,
            "strategy": "balanced_block_chronological",
            "train_blocks": "|".join(str(b) for b in _blocks("train")),
            "val_blocks": "|".join(str(b) for b in _blocks("val")),
            "test_blocks": "|".join(str(b) for b in _blocks("test")),
            "train_off": _n("train", "off"),
            "train_on": _n("train", "on"),
            "val_off": _n("val", "off"),
            "val_on": _n("val", "on"),
            "test_off": _n("test", "off"),
            "test_on": _n("test", "on"),
        }
    )
if manifest_rows:
    manifest_df = pl.DataFrame(manifest_rows)
    manifest_csv = OUT / "split_manifest.csv"
    manifest_df.write_csv(str(manifest_csv))
    print(f"Wrote split manifest -> {manifest_csv}")
    print(manifest_df)

In [ ]:
# Enrich removal details with session label
_sess_lookup = {(s.participant, s.session): s.label for s in session_specs}
for d in all_removal_details:
    d["label"] = _sess_lookup.get((d["participant"], d["session"]), "")

# --- Collect per-split DBS counts --- derived from split parquets directly.
split_dbs_rows = []
for s in session_specs:
    fw = s.psid_variant.split("_")[0]
    base = results_root / fw / s.psid_variant / "split"
    for split_name in ("train", "val", "test"):
        df = pl.read_parquet(base / f"{split_name}.parquet", columns=["stim"])
        n_off = df.filter(pl.col("stim") == "off").height
        n_on = df.filter(pl.col("stim") == "on").height
        split_dbs_rows.append(
            {
                "session": s.label,
                "split": split_name,
                "OFF": n_off,
                "ON": n_on,
                "total": n_off + n_on,
            }
        )
split_dbs_df = pl.DataFrame(split_dbs_rows)
sessions = [s.label for s in session_specs]

session_totals = {
    sl: int(split_dbs_df.filter(pl.col("session") == sl)["total"].sum())
    for sl in sessions
}

split_order = ["train", "val", "test"]
SPLIT_COLOR = {"train": "#D32F2F", "val": "#1976D2", "test": "#616161"}
DBS_ALPHA = {"OFF": 0.55, "ON": 1.00}

reason_meta = [
    ("fragmented", "#E24B4A"),
    ("protocol/events", "#888780"),
    ("plateau (>2.0s)", COLOR_DBS_OFF),
]


def _split_dbs_values(sp, dbs, as_fraction):
    out = []
    for sl in sessions:
        r = split_dbs_df.filter((pl.col("session") == sl) & (pl.col("split") == sp))
        c = r[dbs][0] if r.height else 0
        if as_fraction:
            out.append(c / session_totals[sl] if session_totals[sl] else 0)
        else:
            out.append(c)
    return np.array(out, dtype=float)


# ── Figure 1: Trials removed by reason ───────────────────────────────────
fig_a, ax_a = plt.subplots(figsize=(6.5, 3.2))
bottoms = np.zeros(len(sessions))
for reason, color in reason_meta:
    counts = np.array(
        [
            sum(
                1
                for d in all_removal_details
                if d["label"] == sl and d["reason"] == reason
            )
            for sl in sessions
        ],
        dtype=float,
    )
    bars = ax_a.bar(
        sessions, counts, bottom=bottoms, color=color, width=0.55, label=reason
    )
    stack_bar_label(ax_a, bars, fmt="{:.0f}", min_value=1)
    bottoms += counts
ax_a.set_ylabel("Trials")
ax_a.set_ylim(bottom=0)
ax_a.legend(title="Removal reason")
panel_label(ax_a, "A", "Trials removed by reason")
fig_a.savefig(str(OUT / "fig_trials_removed.png"))
plt.show()

# ── Figure 2: Split ratios with DBS condition ─────────────────────────────
DBS_SERIES = [("OFF", None), ("ON", None)]

fig_b, ax_b = plt.subplots(figsize=(6.5, 3.5))
fig_b.set_constrained_layout(False)
bottoms = np.zeros(len(sessions))
for sp in split_order:
    for dbs, _ in DBS_SERIES:
        vals_frac = _split_dbs_values(sp, dbs, as_fraction=True)
        vals_abs = _split_dbs_values(sp, dbs, as_fraction=False)
        bars = ax_b.bar(
            sessions,
            vals_frac,
            bottom=bottoms,
            color=hex_to_rgba(SPLIT_COLOR[sp], DBS_ALPHA[dbs]),
            width=0.55,
            label=f"{sp} DBS-{dbs}",
        )
        stack_bar_label(ax_b, bars, values=vals_abs, fmt="{:.0f}", min_value=0.02)
        bottoms += vals_frac
ax_b.set_ylabel("Split ratio")
ax_b.set_ylim(0, 1.05)
panel_label(ax_b, "B", "Split ratio with DBS condition")
_h, _l = ax_b.get_legend_handles_labels()
_order = [0, 2, 4, 1, 3, 5]
fig_b.legend(
    [_h[i] for i in _order], [_l[i] for i in _order], ncol=3, loc="lower center"
)
fig_b.subplots_adjust(bottom=0.22)
fig_b.savefig(str(OUT / "fig_split_ratio.png"))
plt.show()

# ── Figure 3: Trial counts per split ─────────────────────────────────────
fig_c, ax_c = plt.subplots(figsize=(6.5, 3.5))
fig_c.set_constrained_layout(False)
bottoms = np.zeros(len(sessions))
for sp in split_order:
    for dbs, _ in DBS_SERIES:
        vals = _split_dbs_values(sp, dbs, as_fraction=False)
        bars = ax_c.bar(
            sessions,
            vals,
            bottom=bottoms,
            color=hex_to_rgba(SPLIT_COLOR[sp], DBS_ALPHA[dbs]),
            width=0.55,
            label=f"{sp} DBS-{dbs}",
        )
        stack_bar_label(ax_c, bars, fmt="{:.0f}", min_value=1)
        bottoms += vals
ax_c.set_ylabel("Trials")
panel_label(ax_c, "C", "Trial counts per split")
_h, _l = ax_c.get_legend_handles_labels()
_order = [0, 2, 4, 1, 3, 5]
fig_c.legend(
    [_h[i] for i in _order], [_l[i] for i in _order], ncol=3, loc="lower center"
)
fig_c.subplots_adjust(bottom=0.22)
fig_c.savefig(str(OUT / "fig_split_counts.png"))
plt.show()

## PSD DBS comparison — all channels per session

In [ ]:
FS_SPLIT = 200
MARGIN_S = 2
MARGIN_SAMP = MARGIN_S * FS_SPLIT
FREQ_CUTOFF = 85

ECOG_CHANNELS = [1, 2, 3, 4]
LAP_CHANNEL_NAMES = ["8-10", "9-11", "10-12", "11-13", "12-14", "13-15", "14-16"]

ECOG_COLORS = [cm.Blues(v) for v in np.linspace(0.40, 0.85, len(ECOG_CHANNELS))]
LAP_COLORS = [cm.Oranges(v) for v in np.linspace(0.35, 0.90, len(LAP_CHANNEL_NAMES))]
ALPHA_OFF = 0.35

## PSD DBS comparison — per session, ECoG + Laplacian overlaid

Each session: left panel = all 4 ECoG channels; right = all 7 Laplacian channels.
Solid lines = DBS-ON (mean across trials), faded = DBS-OFF. Blue shades = ECoG; orange shades = Laplacian.

In [ ]:
# Per-session PSD: all ECoG channels (left) + all Laplacian channels (right).
_SESSION_FIG_IDX = {"PDI1_S2": 2, "PDI1_S4": 3, "PDI4_S2": 4, "PDI4_S3": 5}

for s in session_specs:
    fig, (ax_ecog, ax_lap) = plt.subplots(1, 2, figsize=(9.0, 4.0))
    fig.set_constrained_layout(False)

    # ECoG — 4 channels, blue shades; ON full alpha, OFF faded; all solid
    for ch_idx, ch in enumerate(ECOG_CHANNELS):
        color = ECOG_COLORS[ch_idx]
        m_on = psds_ecog[ch][s.label]["on"]
        m_off = psds_ecog[ch][s.label]["off"]
        if m_on is not None:
            ax_ecog.plot(
                freqs_plot, m_on, color=color, lw=0.9, ls="-", label=f"ECOG_{ch}"
            )
        if m_off is not None:
            ax_ecog.plot(
                freqs_plot, m_off, color=color, lw=0.9, ls="-", alpha=ALPHA_OFF
            )

    ax_ecog.set_xlim(0, FREQ_CUTOFF)
    ax_ecog.set_xticks(range(0, FREQ_CUTOFF + 1, 10))
    ax_ecog.set_xlabel("Frequency (Hz)")
    ax_ecog.set_ylabel("PSD (dB/Hz)")
    panel_label(ax_ecog, "A", f"ECoG - {s.label}")

    # Laplacian — 7 channels, orange shades; ON full, OFF faded; all solid
    for lap_idx, lap_name in enumerate(LAP_CHANNEL_NAMES):
        color = LAP_COLORS[lap_idx]
        m_on = psds_lap[lap_name][s.label]["on"]
        m_off = psds_lap[lap_name][s.label]["off"]
        if m_on is not None:
            ax_lap.plot(freqs_plot, m_on, color=color, lw=0.9, ls="-", label=lap_name)
        if m_off is not None:
            ax_lap.plot(freqs_plot, m_off, color=color, lw=0.9, ls="-", alpha=ALPHA_OFF)

    ax_lap.set_xlim(0, FREQ_CUTOFF)
    ax_lap.set_xticks(range(0, FREQ_CUTOFF + 1, 10))
    ax_lap.set_xlabel("Frequency (Hz)")
    ax_lap.set_ylabel("PSD (dB/Hz)")
    panel_label(ax_lap, "B", f"Laplacian LFP - {s.label}")

    # ECoG legend below panel A: channels + DBS-ON/OFF indicators
    _ch_h, _ch_l = ax_ecog.get_legend_handles_labels()
    _cond_h = [
        Line2D([0], [0], color="gray", lw=1.2, ls="-", label="DBS-ON"),
        Line2D(
            [0], [0], color="gray", lw=1.2, ls="-", alpha=ALPHA_OFF, label="DBS-OFF"
        ),
    ]
    ax_ecog.legend(
        handles=_ch_h + _cond_h,
        labels=_ch_l + ["DBS-ON", "DBS-OFF"],
        loc="upper center",
        bbox_to_anchor=(0.5, -0.18),
        ncol=3,
    )
    # Laplacian legend below panel B
    ax_lap.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=4)

    fig.subplots_adjust(left=0.08, right=0.98, top=0.92, bottom=0.32, wspace=0.30)
    fig_num = _SESSION_FIG_IDX[s.label]
    out_name = f"fig_{fig_num:03d}_psd_{s.label}.png"
    fig.savefig(str(OUT / out_name))
    plt.show()
    print(f"{s.label} -> {out_name}")

## Behavioral signals — raw trial traces (velocity, acceleration)


In [ ]:
# Behavioral raw data — z-scored per trial, mean ± SEM bands (smoothed)

FS_BEH = 200
TRIAL_SEC = 9
N_SAMP = TRIAL_SEC * FS_BEH
SMOOTH_WIN = 41
beh_vars = ["tracing_velocity_x", "tracing_acceleration_magnitude"]


def _smooth(arr, win):
    kernel = np.ones(win) / win
    padded = np.pad(arr, win // 2, mode="edge")
    return np.convolve(padded, kernel, mode="valid")[: len(arr)]


for var_name in beh_vars:
    traces_by_session = {}
    for s in session_specs:
        framework = s.psid_variant.split("_")[0]
        base = results_root / framework / s.psid_variant / "split"
        traces = {"off": [], "on": []}
        for split in ("train", "val", "test"):
            fp = base / f"{split}.parquet"
            if not fp.exists():
                continue
            df = pl.read_parquet(fp, columns=["stim", var_name])
            for row in df.iter_rows(named=True):
                cond = "off" if row["stim"] in ("off", "0") else "on"
                sig = np.array(row[var_name], dtype=float)[:N_SAMP]
                if len(sig) < N_SAMP:
                    continue
                mu, sd = np.nanmean(sig), np.nanstd(sig)
                if sd < 1e-8:
                    continue
                sig = (sig - mu) / sd
                sig = np.nan_to_num(sig, nan=0.0)
                traces[cond].append(sig)
        traces_by_session[s.label] = traces

    fig, axes = plt.subplots(2, 2, sharex=True, sharey=True, figsize=(7.5, 4.8))
    fig.set_constrained_layout(False)
    t_axis = np.arange(N_SAMP) / FS_BEH
    for s_idx, s in enumerate(session_specs):
        ax = axes.flat[s_idx]
        traces = traces_by_session[s.label]
        for cond_key, col_c, cond_label in [
            ("off", COLOR_DBS_OFF, "DBS-OFF"),
            ("on", COLOR_DBS_ON, "DBS-ON"),
        ]:
            cond_traces = traces[cond_key]
            if not cond_traces:
                continue
            mat = np.vstack(cond_traces)
            mean = _smooth(mat.mean(axis=0), SMOOTH_WIN)
            sem = _smooth(mat.std(axis=0) / np.sqrt(len(mat)), SMOOTH_WIN)
            ax.fill_between(
                t_axis, mean - sem, mean + sem, color=col_c, alpha=0.20, linewidth=0
            )
            ax.plot(t_axis, mean, color=col_c, label=cond_label)
        ax.set_xlim(0, TRIAL_SEC)
        ax.set_xticks(range(0, TRIAL_SEC + 1))
        panel_label(ax, chr(65 + s_idx), s.label)

    for ax in axes[-1, :]:
        ax.set_xlabel("Time (s)")
    for ax in axes[:, 0]:
        ax.set_ylabel("z-score")

    handles, labels = axes.flat[0].get_legend_handles_labels()
    fig.legend(handles, labels, ncol=2, loc="lower center", frameon=False)
    fig.subplots_adjust(bottom=0.12)

    safe_name = var_name.replace(" ", "_")
    fig.savefig(str(OUT / f"fig_beh_{safe_name}.png"))
    plt.show()
    print(f"{var_name} figure saved")

## Grid alignment — behavioral resampling to 200 Hz neural grid

**A** Interpolation on the common time grid (100 ms window).  
**B** x position density: raw vs aligned (KDE, pooled across sessions).  
**C** Sample tracing trajectory (single trial).

In [ ]:
# Grid alignment — three standalone figures

GRID_DATA_ROOT = Path(
    "resampled_recordings/participants_at_200Hz_scaled_1e6_narrow_band"
)
MARGIN_SAMP = 400
# Raw behavioural = DPAD-warm, aligned = PSID-blue. Neural dot uses a brighter red
# to distinguish it from the DBS-ON red elsewhere.
RAW_COLOR, ALIGNED_COLOR, NEURAL_COLOR = COLOR_DPAD, COLOR_PSID, "#C43A31"